In [18]:
import os 
import re
import joblib
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report



In [11]:
df = pd.read_csv("../data/phishing_email.csv")
df.info(), df.head()

<class 'pandas.DataFrame'>
RangeIndex: 82486 entries, 0 to 82485
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   text_combined  82486 non-null  str  
 1   label          82486 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.3 MB


(None,
                                        text_combined  label
 0  hpl nom may 25 2001 see attached file hplno 52...      0
 1  nom actual vols 24 th forwarded sabrae zajac h...      0
 2  enron actuals march 30 april 1 201 estimated a...      0
 3  hpl nom may 30 2001 see attached file hplno 53...      0
 4  hpl nom june 1 2001 see attached file hplno 60...      0)

Df has 2 columns : text_combined, label (0 - safe, 1 - phishing)
First, we should clean the data (Exclude numbers, spaces, etc..)

In [13]:
# Preprocessing
def clean_email_text(text):
    if not isinstance(text, str):
        return ""
    # small cases letters
    text = text.lower()
    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # remove double spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text 

df['clean_text'] = df['text_combined'].apply(clean_email_text)
df
    

,text_combined,label,clean_text
0,hpl nom may 25 2001 see attached file hplno 52...,0,hpl nom may see attached file hplno xls hplno xls
1,nom actual vols 24 th forwarded sabrae zajac h...,0,nom actual vols th forwarded sabrae zajac hou ...
2,enron actuals march 30 april 1 201 estimated a...,0,enron actuals march april estimated actuals ma...
3,hpl nom may 30 2001 see attached file hplno 53...,0,hpl nom may see attached file hplno xls hplno xls
4,hpl nom june 1 2001 see attached file hplno 60...,0,hpl nom june see attached file hplno xls hplno...
...,...,...,...
82481,info advantageapartmentscom infoadvantageapart...,1,info advantageapartmentscom infoadvantageapart...
82482,monkeyorg helpdeskmonkeyorg monkeyorg hi josep...,1,monkeyorg helpdeskmonkeyorg monkeyorg hi josep...
82483,help center infohelpcentercoza_infohelpcenterc...,1,help center infohelpcentercozainfohelpcenterco...
82484,metamask infosofamekarcom verify metamask wall...,1,metamask infosofamekarcom verify metamask wall...


In [14]:
# Vectorizing
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

In [16]:
# Training using Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98      7935
           1       0.99      0.98      0.99      8563

    accuracy                           0.98     16498
   macro avg       0.98      0.98      0.98     16498
weighted avg       0.98      0.98      0.98     16498



In [19]:
# Export

os.makedirs('../backend/model', exist_ok=True)

joblib.dump(model, '../backend/model/phishing_model.pkl')
joblib.dump(vectorizer, '../backend/model/vectorizer.pkl')

['../backend/model/vectorizer.pkl']